# ETL Pipeline Examples with th2etl

This notebook provides three examples of how to create and manage ETL pipelines using the `th2etl` client. Each example includes an explanation of the pipeline's purpose, a diagram of its structure, and the code needed to implement it.

**Prerequisites:**

*   The `th2etl` API server is running.
*   The `th2etl` package is installed in your environment.
*   You have a PostgreSQL database configured as described in the main `README.md`.

## Example 1: Simple Sequential Pipeline

This example demonstrates a basic ETL pipeline with a linear, sequential flow.

1.  **Load:** Data is loaded from a CSV file.
2.  **Transform:** The data is processed by a transformer.
3.  **Export:** The transformed data is saved to a PostgreSQL table.

Each step runs only after the previous one has successfully completed.

**Execution Flow Diagram:**

```mermaid
graph TD
    subgraph Stage 1
        Load[cars_csv_loader]
    end

    subgraph Stage 2
        Transform[cars_transformer]
    end
    
    subgraph Stage 3
        Export[cars_exporter]
    end

    Load --> Transform
    Transform --> Export
```


In [ ]:
from th2etl.helpers.th2etl_client import Th2etlClient

# Initialize the client
client = Th2etlClient()

# 1. Create the blocs
client.create_bloc(
    name="cars_csv_loader",
    bloc_type="csv_loader",
    config={"file_path": "datasets/cars.csv"}
)

client.create_bloc(
    name="cars_transformer",
    bloc_type="example_transformer",
    config={"factor": 10}
)

client.create_bloc(
    name="cars_exporter",
    bloc_type="postgres_exporter",
    config={"table_name": "cars_data", "source_bloc": "cars_transformer"}
)

# 2. Create the pipeline with sequential stages
client.create_pipeline(
    name="sequential_pipeline",
    stages=[
        ["cars_csv_loader"],
        ["cars_transformer"],
        ["cars_exporter"]
    ]
)

print("Simple sequential pipeline created successfully.")


## Example 2: Parallel Transformation Pipeline

This example showcases a pipeline that performs multiple transformations on the same source data in parallel.

1.  **Load:** Data is loaded from a single source.
2.  **Transform (in parallel):** Two different transformers process the source data concurrently.
3.  **Export (in parallel):** The results of each transformation are saved to separate tables.

This is useful for when you need to perform multiple independent operations on the same dataset, as it can significantly speed up the total execution time.

**Execution Flow Diagram:**

```mermaid
graph TD
    subgraph Stage 1: Load
        Load[api_loader]
    end

    subgraph Stage 2: Transform (Parallel)
        TransformA[transformer_a]
        TransformB[transformer_b]
    end
    
    subgraph Stage 3: Export (Parallel)
        ExportA[exporter_a]
        ExportB[exporter_b]
    end

    Load --> TransformA
    Load --> TransformB
    TransformA --> ExportA
    TransformB --> ExportB
```


In [ ]:
# 1. Create the blocs
client.create_bloc(
    name="api_loader",
    bloc_type="api_loader",
    config={"url": "https://jsonplaceholder.typicode.com/posts"}
)

client.create_bloc(
    name="transformer_a",
    bloc_type="example_transformer",
    config={"factor": 2}
)

client.create_bloc(
    name="transformer_b",
    bloc_type="example_transformer",
    config={"factor": -1}
)

client.create_bloc(
    name="exporter_a",
    bloc_type="postgres_exporter",
    config={"table_name": "transformed_a", "source_bloc": "transformer_a"}
)

client.create_bloc(
    name="exporter_b",
    bloc_type="postgres_exporter",
    config={"table_name": "transformed_b", "source_bloc": "transformer_b"}
)

# 2. Create the pipeline with parallel stages
client.create_pipeline(
    name="parallel_transform_pipeline",
    stages=[
        ["api_loader"],
        ["transformer_a", "transformer_b"],
        ["exporter_a", "exporter_b"]
    ]
)

print("Parallel transformation pipeline created successfully.")


## Example 3: Multi-Source Merge Pipeline

This example shows a more complex pipeline that loads data from two different sources in parallel, then merges the results in a later stage.

1.  **Load (in parallel):** Data is loaded from a CSV file and a web API at the same time.
2.  **Merge:** A transformer (which you would need to implement) takes the output of both loaders and merges them.
3.  **Export:** The merged data is saved.

This pattern is useful for enriching a primary data source with additional information from another system.

**Execution Flow Diagram:**

```mermaid
graph TD
    subgraph Stage 1: Load (Parallel)
        LoadA[csv_source]
        LoadB[api_source]
    end

    subgraph Stage 2: Merge
        Merge[merge_data]
    end
    
    subgraph Stage 3: Export
        Export[final_export]
    end

    LoadA --> Merge
    LoadB --> Merge
    Merge --> Export
```


In [ ]:
# Note: This example assumes a "merge_transformer" bloc exists.
# You would need to implement a new TransformerBloc that can take two
# source_blocs and merge their data.

# 1. Create the blocs
client.create_bloc(
    name="csv_source",
    bloc_type="csv_loader",
    config={"file_path": "datasets/cars.csv"}
)

client.create_bloc(
    name="api_source",
    bloc_type="api_loader",
    config={"url": "https://jsonplaceholder.typicode.com/users"}
)

# This is a placeholder for a custom merge bloc you would create.
client.create_bloc(
    name="merge_data",
    bloc_type="example_transformer", # Replace with your custom merge transformer
    config={
        "source_a": "csv_source",
        "source_b": "api_source",
        "merge_key": "id"
    }
)

client.create_bloc(
    name="final_export",
    bloc_type="postgres_exporter",
    config={"table_name": "merged_data", "source_bloc": "merge_data"}
)

# 2. Create the pipeline with a parallel load stage
client.create_pipeline(
    name="multi_source_merge_pipeline",
    stages=[
        ["csv_source", "api_source"],
        ["merge_data"],
        ["final_export"]
    ]
)

print("Multi-source merge pipeline created successfully.")
